# Baseline Defender

## Stage 1 : Initial set up

In [4]:
import pandas as pd
import numpy as np

DATA_PATH = "..\synthetic_transactions_v1.csv"

# time split ratio
TRAIN_FRAC = 0.7

Load CSV

In [5]:
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()


(201892, 11)


,Transaction_id,Timestamp,user_id,Card_id,home_country,Transaction amount,merchant_id,merchant_category,merchant_country,Channel,is_fraud
0,1,2025-01-01 07:47:17,USER_C_1001,CARD_C_1001_A,GB,4.00,MERCH_PRET_03,coffee,GB,POS,0
1,2,2025-01-01 07:47:06,USER_C_1001,CARD_C_1001_A,GB,0.90,MERCH_TFL_01,transport_public,GB,POS,0
2,3,2025-01-01 08:57:34,USER_C_1001,CARD_C_1001_A,GB,4.79,MERCH_PRET_03,coffee,GB,POS,0
3,4,2025-01-01 09:27:02,USER_C_1001,CARD_C_1001_A,GB,4.94,MERCH_PRET_03,coffee,GB,POS,0
4,5,2025-01-01 10:13:14,USER_C_1001,CARD_C_1001_A,GB,4.99,MERCH_PRET_03,coffee,GB,POS,0


In [7]:
# Parse timestamps
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

# Drop bad timestamps if any
df = df.dropna(subset=['Timestamp'])

# Sort by timestamp 
df = df.sort_values(['Timestamp', 'user_id']).reset_index(drop=True)

df[["Timestamp", "user_id", "merchant_category", "Transaction amount", "is_fraud"]].head()

,Timestamp,user_id,merchant_category,Transaction amount,is_fraud
0,2025-01-01 00:02:36,USER_S_2147,transport_ride_hail,10.78,0
1,2025-01-01 00:04:00,USER_S_2004,transport_ride_hail,20.96,0
2,2025-01-01 00:05:59,USER_S_2006,transport_ride_hail,19.31,0
3,2025-01-01 00:06:33,USER_S_2101,transport_ride_hail,14.02,0
4,2025-01-01 00:07:40,USER_S_2099,transport_ride_hail,7.98,0


Quick data check

In [8]:
required_cols = [
    "Transaction_id", "Timestamp", "user_id", "Card_id",
    "Transaction amount", "merchant_id", "merchant_category",
    "merchant_country", "Channel", "is_fraud"
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing columns:", missing)

print("Fraud rate:", df["is_fraud"].mean())
print("Users:", df["user_id"].nunique())
print("Date range:", df["Timestamp"].min(), "->", df["Timestamp"].max())

Missing columns: []
Fraud rate: 0.0
Users: 400
Date range: 2025-01-01 00:02:36 -> 2025-03-31 23:59:36


Splitting of data

In [9]:
# pick cutoff by position in time-sorted rows
cut_idx = int(len(df) * TRAIN_FRAC)
cut_time = df.loc[cut_idx, "Timestamp"]

train_df = df[df["Timestamp"] < cut_time].copy()
test_df  = df[df["Timestamp"] >= cut_time].copy()

print("Cut time:", cut_time)
print("Train:", train_df.shape, train_df["Timestamp"].min(), "->", train_df["Timestamp"].max())
print("Test :", test_df.shape,  test_df["Timestamp"].min(),  "->", test_df["Timestamp"].max())


Cut time: 2025-03-05 04:17:34
Train: (141324, 11) 2025-01-01 00:02:36 -> 2025-03-05 04:16:59
Test : (60568, 11) 2025-03-05 04:17:34 -> 2025-03-31 23:59:36


In [10]:
# Ensure chronological separation
assert train_df["Timestamp"].max() < test_df["Timestamp"].min()

# Ensure user overlap (realistic: same users appear in both periods)
overlap_users = len(set(train_df["user_id"]) & set(test_df["user_id"]))
print("Users in both train & test:", overlap_users, "/", df["user_id"].nunique())


Users in both train & test: 400 / 400


In [11]:
train_df.to_csv("train_transactions.csv", index=False)
test_df.to_csv("test_transactions.csv", index=False)

## Stage 2 : Feature Engineering